In [ ]:
from Bio import Entrez
import pandas as pd

Entrez.email = "samireformed786@gmail.com"

def search_geo(query, retmax=500):
    handle = Entrez.esearch(db="gds", term=query, retmax=retmax, sort="pub+date")
    record = Entrez.read(handle)
    handle.close()
    return record['IdList']

def fetch_summaries(id_list):
    if not id_list:          # guard against empty list
        return []
    ids = ",".join(id_list)
    handle = Entrez.esummary(db="gds", id=ids)
    records = Entrez.read(handle)
    handle.close()
    return records

def to_dataframe(summaries):
    rows = []
    for s in summaries:
        rows.append({
            'accession':  s.get('Accession', ''),
            'title':      s.get('title', ''),
            'organism':   s.get('taxon', ''),
            'n_samples':  int(s.get('n_samples', 0)),
            'date':       s.get('PDAT', ''),
            'has_pub':    len(s.get('PubMedIds', [])) > 0,
            'type':       s.get('gdsType', ''),
            'summary':    s.get('summary', ''),
        })
    return pd.DataFrame(rows)

In [ ]:
# ── PLANTS RELEVANT TO BANGLADESH ──────────────────────────────────────────

bangladesh_crops = [
    # Staple crops
    "Oryza sativa[organism] AND stress",           # rice — blast, flood, salinity
    "Oryza sativa[organism] AND submergence",
    "Oryza sativa[organism] AND bacterial blight",
    
    # Vegetables common in BD
    "Solanum melongena[organism]",                 # brinjal/eggplant — very few datasets
    "Trichosanthes dioica[organism]",              # pointed gourd — almost nothing
    "Momordica charantia[organism]",               # bitter gourd
    "Lagenaria siceraria[organism]",               # bottle gourd
    
    # Cash crops
    "Corchorus olitorius[organism]",               # jute — Bangladesh's own crop, very sparse
    "Corchorus capsularis[organism]",              # white jute
    
    # Pulses
    "Lens culinaris[organism]",                    # lentil
    "Vigna mungo[organism]",                       # black gram
    "Vigna radiata[organism]",                     # mung bean
    
    # Climate stress — very relevant to BD
    "rice AND salinity stress AND transcriptome",
    "rice AND flood tolerance AND RNA-seq",
    "wheat AND heat stress AND Bangladesh",
]

# ── HUMAN DISEASES COMMON IN BANGLADESH ────────────────────────────────────

bangladesh_diseases = [
    "Vibrio cholerae[organism]",                   # cholera — endemic
    "Mycobacterium tuberculosis[organism] AND RNA-seq",  # TB burden
    "dengue virus[organism] AND transcriptome",    # dengue
    "Leishmania donovani[organism]",               # kala-azar — BD is endemic zone
    "arsenic AND human AND gene expression",       # groundwater arsenic poisoning
    "malnutrition AND child AND transcriptome",    
    "Helicobacter pylori[organism] AND gastric",   # very high prevalence in BD
]

human_diseases = [

    # ── INFECTIOUS / BACTERIAL ─────────────────────────────────────────────
    "Staphylococcus aureus[organism] AND RNA-seq",
    "MRSA AND transcriptome",
    "Streptococcus pneumoniae[organism] AND gene expression",
    "Streptococcus pyogenes[organism] AND transcriptome",
    "Klebsiella pneumoniae[organism] AND RNA-seq",
    "Pseudomonas aeruginosa[organism] AND transcriptome",
    "Acinetobacter baumannii[organism] AND gene expression",
    "Clostridioides difficile[organism] AND transcriptome",
    "Salmonella enterica[organism] AND RNA-seq",
    "Escherichia coli[organism] AND pathogenic AND transcriptome",
    "Neisseria gonorrhoeae[organism] AND gene expression",
    "Neisseria meningitidis[organism] AND transcriptome",
    "Bordetella pertussis[organism] AND RNA-seq",
    "Listeria monocytogenes[organism] AND transcriptome",
    "Campylobacter jejuni[organism] AND gene expression",
    "Treponema pallidum[organism] AND transcriptome",
    "Borrelia burgdorferi[organism] AND RNA-seq",
    "Chlamydia trachomatis[organism] AND transcriptome",
    "Rickettsia[organism] AND gene expression",
    "Burkholderia pseudomallei[organism] AND transcriptome",
    "Francisella tularensis[organism] AND RNA-seq",
    "Yersinia pestis[organism] AND transcriptome",
    "Brucella[organism] AND gene expression",
    "Leptospira[organism] AND transcriptome",
    "Haemophilus influenzae[organism] AND RNA-seq",

    # ── INFECTIOUS / MYCOBACTERIAL & FUNGAL ────────────────────────────────
    "Mycobacterium leprae[organism] AND transcriptome",
    "Mycobacterium abscessus[organism] AND RNA-seq",
    "Aspergillus fumigatus[organism] AND transcriptome",
    "Candida albicans[organism] AND gene expression",
    "Candida auris[organism] AND transcriptome",
    "Cryptococcus neoformans[organism] AND RNA-seq",
    "Histoplasma capsulatum[organism] AND transcriptome",
    "Coccidioides[organism] AND gene expression",
    "Mucor[organism] AND transcriptome",

    # ── INFECTIOUS / PARASITIC ─────────────────────────────────────────────
    "Plasmodium falciparum[organism] AND transcriptome",
    "Plasmodium vivax[organism] AND RNA-seq",
    "Toxoplasma gondii[organism] AND gene expression",
    "Trypanosoma brucei[organism] AND transcriptome",
    "Trypanosoma cruzi[organism] AND RNA-seq",
    "Leishmania major[organism] AND transcriptome",
    "Leishmania braziliensis[organism] AND gene expression",
    "Schistosoma mansoni[organism] AND RNA-seq",
    "Schistosoma haematobium[organism] AND transcriptome",
    "Giardia lamblia[organism] AND gene expression",
    "Entamoeba histolytica[organism] AND transcriptome",
    "Trichomonas vaginalis[organism] AND RNA-seq",
    "Cryptosporidium parvum[organism] AND transcriptome",
    "Wuchereria bancrofti[organism] AND gene expression",
    "Onchocerca volvulus[organism] AND transcriptome",

    # ── INFECTIOUS / VIRAL ─────────────────────────────────────────────────
    "SARS-CoV-2 AND host response AND RNA-seq",
    "influenza AND human AND transcriptome",
    "HIV AND latency AND gene expression",
    "HIV AND reservoir AND RNA-seq",
    "Epstein-Barr virus AND transcriptome",
    "cytomegalovirus AND human AND gene expression",
    "hepatitis B virus AND liver AND transcriptome",
    "hepatitis C virus AND RNA-seq",
    "herpes simplex virus AND transcriptome",
    "varicella zoster virus AND gene expression",
    "human papillomavirus AND transcriptome",
    "norovirus AND human AND gene expression",
    "rotavirus AND intestinal AND transcriptome",
    "respiratory syncytial virus AND RNA-seq",
    "Zika virus AND neural AND transcriptome",
    "West Nile virus AND gene expression",
    "chikungunya AND transcriptome",
    "Ebola virus AND host AND RNA-seq",
    "Marburg virus AND transcriptome",
    "rabies virus AND neural AND gene expression",
    "measles virus AND transcriptome",
    "mumps virus AND gene expression",
    "adenovirus AND human AND transcriptome",
    "parvovirus B19 AND gene expression",
    "human metapneumovirus AND transcriptome",

    # ── CANCER / SOLID TUMORS ──────────────────────────────────────────────
    "glioblastoma AND single cell AND RNA-seq",
    "glioblastoma AND recurrence AND transcriptome",
    "glioma AND tumor microenvironment AND RNA-seq",
    "medulloblastoma AND transcriptome",
    "neuroblastoma AND differentiation AND RNA-seq",
    "lung adenocarcinoma AND early stage AND transcriptome",
    "lung squamous cell carcinoma AND RNA-seq",
    "small cell lung cancer AND transcriptome",
    "breast cancer AND triple negative AND RNA-seq",
    "breast cancer AND metastasis AND transcriptome",
    "breast cancer AND resistance AND gene expression",
    "pancreatic ductal adenocarcinoma AND transcriptome",
    "pancreatic cancer AND stroma AND RNA-seq",
    "colorectal cancer AND microsatellite AND transcriptome",
    "colorectal cancer AND liver metastasis AND RNA-seq",
    "gastric cancer AND diffuse type AND transcriptome",
    "gastric cancer AND H. pylori AND gene expression",
    "hepatocellular carcinoma AND cirrhosis AND RNA-seq",
    "cholangiocarcinoma AND transcriptome",
    "ovarian cancer AND high grade AND RNA-seq",
    "ovarian cancer AND chemoresistance AND transcriptome",
    "cervical cancer AND HPV AND gene expression",
    "endometrial cancer AND transcriptome",
    "prostate cancer AND castration resistant AND RNA-seq",
    "prostate cancer AND bone metastasis AND transcriptome",
    "bladder cancer AND muscle invasive AND RNA-seq",
    "renal cell carcinoma AND clear cell AND transcriptome",
    "thyroid cancer AND papillary AND RNA-seq",
    "adrenocortical carcinoma AND transcriptome",
    "osteosarcoma AND RNA-seq",
    "Ewing sarcoma AND transcriptome",
    "rhabdomyosarcoma AND gene expression",
    "liposarcoma AND transcriptome",
    "mesothelioma AND RNA-seq",
    "esophageal squamous cell carcinoma AND transcriptome",
    "head and neck squamous cell carcinoma AND RNA-seq",
    "nasopharyngeal carcinoma AND transcriptome",
    "oral cancer AND gene expression",
    "skin squamous cell carcinoma AND transcriptome",
    "Merkel cell carcinoma AND RNA-seq",

    # ── CANCER / BLOOD & LYMPHOID ──────────────────────────────────────────
    "acute myeloid leukemia AND single cell AND RNA-seq",
    "acute lymphoblastic leukemia AND transcriptome",
    "chronic myeloid leukemia AND blast crisis AND RNA-seq",
    "chronic lymphocytic leukemia AND transcriptome",
    "diffuse large B cell lymphoma AND transcriptome",
    "follicular lymphoma AND transformation AND RNA-seq",
    "mantle cell lymphoma AND transcriptome",
    "Hodgkin lymphoma AND Reed-Sternberg AND gene expression",
    "multiple myeloma AND bone marrow AND RNA-seq",
    "myelodysplastic syndrome AND transcriptome",
    "myeloproliferative neoplasm AND gene expression",
    "T cell lymphoma AND transcriptome",
    "natural killer cell lymphoma AND RNA-seq",

    # ── CARDIOVASCULAR ─────────────────────────────────────────────────────
    "heart failure AND single cell AND RNA-seq",
    "dilated cardiomyopathy AND transcriptome",
    "hypertrophic cardiomyopathy AND gene expression",
    "atrial fibrillation AND transcriptome",
    "myocardial infarction AND RNA-seq",
    "cardiac fibrosis AND transcriptome",
    "atherosclerosis AND plaque AND RNA-seq",
    "aortic stenosis AND transcriptome",
    "pulmonary arterial hypertension AND gene expression",
    "aortic aneurysm AND transcriptome",
    "peripheral artery disease AND RNA-seq",
    "congenital heart disease AND transcriptome",

    # ── NEUROLOGICAL ───────────────────────────────────────────────────────
    "Alzheimer's disease AND single cell AND RNA-seq",
    "Alzheimer's disease AND early onset AND transcriptome",
    "Parkinson's disease AND substantia nigra AND RNA-seq",
    "Parkinson's disease AND gut AND transcriptome",
    "amyotrophic lateral sclerosis AND transcriptome",
    "frontotemporal dementia AND RNA-seq",
    "Huntington's disease AND transcriptome",
    "multiple sclerosis AND lesion AND RNA-seq",
    "epilepsy AND hippocampus AND transcriptome",
    "schizophrenia AND prefrontal cortex AND RNA-seq",
    "bipolar disorder AND transcriptome",
    "major depressive disorder AND RNA-seq",
    "autism spectrum disorder AND transcriptome",
    "attention deficit hyperactivity disorder AND gene expression",
    "traumatic brain injury AND transcriptome",
    "spinal cord injury AND RNA-seq",
    "neuropathic pain AND transcriptome",
    "migraine AND gene expression",
    "prion disease AND transcriptome",
    "spinocerebellar ataxia AND RNA-seq",
    "Rett syndrome AND transcriptome",
    "fragile X syndrome AND gene expression",

    # ── METABOLIC & ENDOCRINE ──────────────────────────────────────────────
    "type 2 diabetes AND islet AND RNA-seq",
    "type 1 diabetes AND pancreas AND transcriptome",
    "obesity AND adipose AND single cell AND RNA-seq",
    "non-alcoholic steatohepatitis AND transcriptome",
    "non-alcoholic fatty liver AND RNA-seq",
    "metabolic syndrome AND gene expression",
    "hypothyroidism AND transcriptome",
    "hyperthyroidism AND gene expression",
    "Cushing syndrome AND transcriptome",
    "polycystic ovary syndrome AND RNA-seq",
    "hyperuricemia AND gout AND gene expression",
    "phenylketonuria AND transcriptome",
    "Wilson disease AND gene expression",

    # ── RESPIRATORY ────────────────────────────────────────────────────────
    "chronic obstructive pulmonary disease AND transcriptome",
    "asthma AND severe AND RNA-seq",
    "idiopathic pulmonary fibrosis AND single cell AND RNA-seq",
    "sarcoidosis AND lung AND transcriptome",
    "bronchiectasis AND gene expression",
    "cystic fibrosis AND airway AND transcriptome",
    "acute respiratory distress syndrome AND RNA-seq",
    "pulmonary embolism AND transcriptome",
    "hypersensitivity pneumonitis AND gene expression",

    # ── GASTROINTESTINAL ───────────────────────────────────────────────────
    "Crohn's disease AND single cell AND RNA-seq",
    "ulcerative colitis AND transcriptome",
    "irritable bowel syndrome AND gene expression",
    "celiac disease AND transcriptome",
    "eosinophilic esophagitis AND RNA-seq",
    "primary sclerosing cholangitis AND transcriptome",
    "primary biliary cholangitis AND gene expression",
    "autoimmune hepatitis AND transcriptome",
    "liver cirrhosis AND RNA-seq",
    "short bowel syndrome AND gene expression",

    # ── RENAL ──────────────────────────────────────────────────────────────
    "chronic kidney disease AND transcriptome",
    "diabetic nephropathy AND single cell AND RNA-seq",
    "IgA nephropathy AND transcriptome",
    "focal segmental glomerulosclerosis AND gene expression",
    "lupus nephritis AND RNA-seq",
    "acute kidney injury AND transcriptome",
    "polycystic kidney disease AND gene expression",
    "renal transplant rejection AND transcriptome",

    # ── MUSCULOSKELETAL ────────────────────────────────────────────────────
    "rheumatoid arthritis AND synovium AND single cell AND RNA-seq",
    "osteoarthritis AND cartilage AND transcriptome",
    "systemic lupus erythematosus AND transcriptome",
    "ankylosing spondylitis AND gene expression",
    "psoriatic arthritis AND transcriptome",
    "systemic sclerosis AND RNA-seq",
    "Sjogren syndrome AND transcriptome",
    "myositis AND gene expression",
    "Duchenne muscular dystrophy AND transcriptome",
    "facioscapulohumeral muscular dystrophy AND RNA-seq",
    "osteoporosis AND bone AND transcriptome",

    # ── SKIN ───────────────────────────────────────────────────────────────
    "psoriasis AND single cell AND RNA-seq",
    "atopic dermatitis AND transcriptome",
    "vitiligo AND gene expression",
    "systemic sclerosis AND skin AND RNA-seq",
    "hidradenitis suppurativa AND transcriptome",
    "melanoma AND immune AND RNA-seq",
    "wound healing AND transcriptome",
    "keloid AND gene expression",

    # ── REPRODUCTIVE & DEVELOPMENTAL ──────────────────────────────────────
    "preeclampsia AND placenta AND RNA-seq",
    "endometriosis AND transcriptome",
    "recurrent pregnancy loss AND gene expression",
    "preterm birth AND transcriptome",
    "male infertility AND testis AND RNA-seq",
    "premature ovarian insufficiency AND transcriptome",
    "gestational diabetes AND gene expression",

    # ── HEMATOLOGICAL (NON-CANCER) ─────────────────────────────────────────
    "sickle cell disease AND transcriptome",
    "thalassemia AND gene expression",
    "aplastic anemia AND transcriptome",
    "hemophilia AND gene expression",
    "thrombocytopenia AND RNA-seq",
    "paroxysmal nocturnal hemoglobinuria AND transcriptome",

    # ── EYE ────────────────────────────────────────────────────────────────
    "age-related macular degeneration AND transcriptome",
    "diabetic retinopathy AND single cell AND RNA-seq",
    "glaucoma AND retinal ganglion AND transcriptome",
    "uveitis AND gene expression",
    "dry eye disease AND transcriptome",

    # ── RARE & ORPHAN DISEASES ─────────────────────────────────────────────
    "Fabry disease AND transcriptome",
    "Gaucher disease AND gene expression",
    "Niemann-Pick disease AND transcriptome",
    "mucopolysaccharidosis AND RNA-seq",
    "Marfan syndrome AND transcriptome",
    "Ehlers-Danlos syndrome AND gene expression",
    "progeria AND transcriptome",
    "Werner syndrome AND RNA-seq",
    "xeroderma pigmentosum AND transcriptome",
    "tuberous sclerosis AND gene expression",
    "neurofibromatosis AND transcriptome",
    "Li-Fraumeni syndrome AND RNA-seq",
    "CHARGE syndrome AND transcriptome",
    "DiGeorge syndrome AND gene expression",
]

rare_disease = [
    # ── RARE & ORPHAN DISEASES ─────────────────────────────────────────────
    "Fabry disease AND transcriptome",
    "Gaucher disease AND gene expression",
    "Niemann-Pick disease AND transcriptome",
    "mucopolysaccharidosis AND RNA-seq",
    "Marfan syndrome AND transcriptome",
    "Ehlers-Danlos syndrome AND gene expression",
    "progeria AND transcriptome",
    "Werner syndrome AND RNA-seq",
    "xeroderma pigmentosum AND transcriptome",
    "tuberous sclerosis AND gene expression",
    "neurofibromatosis AND transcriptome",
    "Li-Fraumeni syndrome AND RNA-seq",
    "CHARGE syndrome AND transcriptome",
    "DiGeorge syndrome AND gene expression",

]

neuro = [
    # ── NEUROLOGICAL ───────────────────────────────────────────────────────
    "Alzheimer's disease AND single cell AND RNA-seq",
    "Alzheimer's disease AND early onset AND transcriptome",
    "Parkinson's disease AND substantia nigra AND RNA-seq",
    "Parkinson's disease AND gut AND transcriptome",
    "amyotrophic lateral sclerosis AND transcriptome",
    "frontotemporal dementia AND RNA-seq",
    "Huntington's disease AND transcriptome",
    "multiple sclerosis AND lesion AND RNA-seq",
    "epilepsy AND hippocampus AND transcriptome",
    "schizophrenia AND prefrontal cortex AND RNA-seq",
    "bipolar disorder AND transcriptome",
    "major depressive disorder AND RNA-seq",
    "autism spectrum disorder AND transcriptome",
    "attention deficit hyperactivity disorder AND gene expression",
    "traumatic brain injury AND transcriptome",
    "spinal cord injury AND RNA-seq",
    "neuropathic pain AND transcriptome",
    "migraine AND gene expression",
    "prion disease AND transcriptome",
    "spinocerebellar ataxia AND RNA-seq",
    "Rett syndrome AND transcriptome",
    "fragile X syndrome AND gene expression",
]
print(f"Total HUMAN DISEASES queries: {len(human_diseases)}")

In [10]:

# run all and collect
all_results = []
# queries_to_run = bangladesh_crops + bangladesh_diseases
queries_to_run = neuro

for q in queries_to_run:
    ids = search_geo(q, retmax=100)
    print(f"  {q[:55]:<55} → {len(ids)} datasets")
    if ids:
        summaries = fetch_summaries(ids[:50])
        df = to_dataframe(summaries)
        df['query'] = q
        all_results.append(df)

master = pd.concat(all_results, ignore_index=True).drop_duplicates('accession')
print(f"\nTotal unique datasets: {len(master)}")




  Alzheimer's disease AND early onset AND transcriptome   → 21 datasets
  Parkinson's disease AND substantia nigra AND RNA-seq    → 35 datasets
  Parkinson's disease AND gut AND transcriptome           → 4 datasets
  amyotrophic lateral sclerosis AND transcriptome         → 100 datasets
  frontotemporal dementia AND RNA-seq                     → 44 datasets
  Huntington's disease AND transcriptome                  → 100 datasets
  multiple sclerosis AND lesion AND RNA-seq               → 14 datasets
  epilepsy AND hippocampus AND transcriptome              → 31 datasets
  schizophrenia AND prefrontal cortex AND RNA-seq         → 24 datasets
  bipolar disorder AND transcriptome                      → 32 datasets
  major depressive disorder AND RNA-seq                   → 29 datasets
  autism spectrum disorder AND transcriptome              → 100 datasets
  attention deficit hyperactivity disorder AND gene expre → 100 datasets
  traumatic brain injury AND transcriptome                → 3

In [11]:
# No publication + enough samples to work with
unpublished = master[
    (master['has_pub']  == False) &
    (master['n_samples'] >= 6)
].sort_values('date', ascending=False)

# print(f"Unpublished datasets: {len(unpublished)}")
# print(unpublished[['accession','title','organism','n_samples','date','query']].to_string())

In [12]:
# Save to CSV
output_file_csv = 'unpublished_geo_datasets_neuro.csv'
unpublished.to_csv(output_file_csv, index=False)
print("Saved to unpublished_geo_datasets.csv")

# # Or just select the columns you actually care about
# cols = ['accession', 'title', 'organism', 'n_samples', 'date', 'type', 'query']
# unpublished[cols].to_csv('unpublished_geo_datasets.csv', index=False)

Saved to unpublished_geo_datasets.csv
